# Real IBM quantum processor
Circuits written in Qiskit, sent to the least busy IBM processor: the Bell state, and Grover's search for each of the 8 possible secrets (so whatever number the audience picks, there is a real-hardware result for it).
The queue can take minutes, so the results are collected later.

In [ ]:
import json, os
from datetime import date
from dotenv import load_dotenv
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

load_dotenv()
service = QiskitRuntimeService(channel="ibm_quantum_platform",
                               token=os.environ["IBM_QUANTUM_PLATFORM_API_KEY"],
                               instance=os.environ.get("IBM_QUANTUM_INSTANCE"))  # optional CRN in .env: skips the instance lookup
backend = service.least_busy(operational=True, simulator=False)
print("backend:", backend.name)

In [ ]:
# Bell state (slide 6 and extra_entanglement.ipynb)
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

# Demo 1: Grover, 3 qubits, 2 iterations, one circuit per possible secret
def ccz(qc):
    qc.h(2); qc.ccx(0, 1, 2); qc.h(2)

def grover_circuit(secret):
    bits = format(secret, "03b")
    zeros = [q for q in range(3) if bits[2 - q] == "0"]   # Qiskit is little-endian: qubit q is bit q from the right
    qc = QuantumCircuit(3)
    qc.h(range(3))
    for _ in range(2):
        if zeros: qc.x(zeros)
        ccz(qc)                                            # oracle: flip the sign of the secret
        if zeros: qc.x(zeros)
        qc.h(range(3)); qc.x(range(3)); ccz(qc); qc.x(range(3)); qc.h(range(3))  # diffusion
    qc.measure_all()
    return qc

names = ["bell"] + [f"grover_{s}" for s in range(8)]
circuits = [bell] + [grover_circuit(s) for s in range(8)]

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa = [pm.run(c) for c in circuits]
print("two-qubit gates after transpiling:", [c.count_ops().get("cz", 0) + c.count_ops().get("ecr", 0) for c in isa])

In [ ]:
job = Sampler(mode=backend).run(isa, shots=1000)
json.dump({"job_id": job.job_id(), "backend": backend.name, "names": names}, open("ibm_job.json", "w"))
print("sent:", job.job_id())

### Collect the results

In [ ]:
# This cell can be run multiple times. First the job is QUEUED. After some time it might be DONE and the data will be overwritten.
saved = json.load(open("ibm_job.json"))
job = service.job(saved["job_id"])
print("status:", job.status())

if not job.done():   # don't wait for the queue: keep the previously saved results
    print("Still running, using the saved ibm_results.json")
else:
    result = job.result()
    results = {name: {"counts": result[i].data.meas.get_counts(),
                      "backend": saved["backend"],
                      "date": date.today().isoformat()}
               for i, name in enumerate(saved.get("names", ["bell", "grover"]))}
    json.dump(results, open("ibm_results.json", "w"), indent=1)
    print(results["bell"]["counts"])
    print("QPU time used:", job.usage(), "s")